# Titanic — Предсказание выживания пассажиров

**Соревнование:** [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)  
**Тема:** Бинарная классификация  
**Инструменты:** `pandas`, `RandomForestClassifier`

---

## О задаче

«Титаник» затонул 15 апреля 1912 года после столкновения с айсбергом. Из 2224 человек на борту выжили только 710. Нужно предсказать, кто из пассажиров выжил (`Survived` — 0/1). Это классическое стартовое соревнование Kaggle.

**Историческая подсказка:** выживание было не случайным. В первую очередь спасали женщин и детей, а у пассажиров первого класса был лучший доступ к шлюпкам. Именно эти закономерности и выучит модель.

## Шаги
1. Загрузка данных
2. Разведочный анализ (EDA)
3. Обработка пропусков
4. Кодирование категориальных признаков
5. Обучение модели
6. Оценка через кросс-валидацию
7. Генерация сабмита

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

train = pd.read_csv('/kaggle/input/titanic/train.csv')
test  = pd.read_csv('/kaggle/input/titanic/test.csv')

print('Размер train:', train.shape)
print('Размер test: ', test.shape)
train.head()

## 2. Разведочный анализ (EDA)

Прежде чем что-то строить — посмотрим на данные и найдём главные сигналы.

In [ ]:
# Общая доля выживших
print('Общая доля выживших:', train['Survived'].mean().round(3))
print()

# Выживание по полу — самый сильный сигнал в этих данных
print('Выживание по полу:')
print(train.groupby('Sex')['Survived'].mean())
print()

# Выживание по классу каюты
print('Выживание по классу (Pclass):')
print(train.groupby('Pclass')['Survived'].mean())

In [ ]:
# Пропуски
print('Пропуски в train:')
print(train.isnull().sum()[train.isnull().sum() > 0])

## 3. Обработка пропусков

- **Age** — пропущено ~20%, заполняем медианой
- **Embarked** — 2 пропуска, заполняем модой (самым частым значением)
- **Fare** — 1 пропуск в тесте, заполняем медианой
- **Cabin** — пропущено ~77%, слишком много дыр, признак не используем

In [ ]:
for df in [train, test]:
    df['Age']      = df['Age'].fillna(train['Age'].median())
    df['Embarked'] = df['Embarked'].fillna(train['Embarked'].mode()[0])
    df['Fare']     = df['Fare'].fillna(train['Fare'].median())

## 4. Кодирование категориальных признаков

Модель понимает только числа — переводим текстовые категории в целые числа.

In [ ]:
for df in [train, test]:
    df['Sex']      = df['Sex'].replace({'female': 1, 'male': 0})
    df['Embarked'] = df['Embarked'].replace({'S': 1, 'C': 2, 'Q': 3})

## 5. Признаки и обучение модели

**Используемые признаки:**
- `Pclass` — класс билета (1/2/3)
- `Sex` — пол (самый сильный предиктор)
- `Age` — возраст
- `SibSp` — число братьев/сестёр/супругов на борту
- `Parch` — число родителей/детей на борту
- `Fare` — цена билета (коррелирует с классом)
- `Embarked` — порт посадки

**Модель:** Random Forest — хорошо работает со смешанными признаками и устойчив к выбросам.

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X      = train[features]
y      = train['Survived']
X_test = test[features]

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)
print('Модель обучена.')

## 6. Оценка через кросс-валидацию

Кросс-валидация делит данные на 5 фолдов: обучается на 4, проверяется на 1 — и так 5 раз. Это даёт честную оценку качества, не трогая тестовую выборку.

In [ ]:
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('Accuracy по фолдам:', scores.round(4))
print(f'Средняя accuracy:  {scores.mean():.4f}')
print(f'Стандартное отклонение: {scores.std():.4f}')

## 7. Генерация сабмита

In [ ]:
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived':    predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv сохранён!')
submission.head()